# Monitoring du Data Drift en Production

## Projet Credit Scoring — Partie 2 (MLOps)

Ce notebook présente la mise en place d’un système de **surveillance de dérive des données (data drift)** pour un modèle de scoring crédit déployé en production.

---

### Objectifs

- Comparer les données de production aux données d’entraînement
- Détecter des dérives potentielles dans les distributions
- Évaluer les risques sur la performance du modèle
- Illustrer une pipeline de monitoring MLOps complète

---

### Contexte

Le modèle de scoring crédit a été :
- entraîné sur un dataset historique (Home Credit)
- déployé via une API FastAPI
- utilisé en production avec stockage des prédictions en base PostgreSQL

Nous analysons ici :
- les données de référence (`reference.parquet`)
- les données issues des appels API récents

In [1]:
from pathlib import Path
import pandas as pd
import json

from credexp.monitoring.drift import (
    load_reference_dataframe,
    load_current_dataframe,
    align_reference_and_current,
)

## Chargement des données

Nous utilisons deux sources :

### 1️ Données de référence
- issues du dataset d'entraînement
- supposées représentatives du comportement "normal"

### 2️ Données de production
- issues des logs API stockés en base PostgreSQL
- reflètent l'utilisation réelle du modèle

In [2]:
from credexp.monitoring.drift import (
    load_reference_dataframe,
    load_current_dataframe,
    align_reference_and_current,
)

reference_df = load_reference_dataframe()
current_df = load_current_dataframe(limit=500)

reference_df, current_df = align_reference_and_current(reference_df, current_df)

reference_df.shape, current_df.shape

{"ts": "2026-04-14T23:17:26Z", "level": "INFO", "name": "credexp.monitoring.drift", "msg": "Loading reference dataframe from G:\\Mon Drive\\OC\\Projet_6\\credexp\\data\\processed\\reference.parquet"}
{"ts": "2026-04-14T23:17:26Z", "level": "INFO", "name": "credexp.data.io", "msg": "load_parquet path=G:\\Mon Drive\\OC\\Projet_6\\credexp\\data\\processed\\reference.parquet"}


((20000, 3), (12, 3))

## Aperçu des données

Nous vérifions que :
- les colonnes sont alignées
- les données sont exploitables

In [3]:
reference_df.head()

,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3
0,0.044202,0.544933,NaN
1,0.543037,0.587365,0.692559
2,NaN,0.643635,NaN
3,NaN,0.746168,0.360613
4,NaN,0.648460,0.486653


In [4]:
current_df.head()

,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3
0,0.62,0.41,0.41
1,0.42,0.41,0.41
2,0.52,0.41,0.41
3,0.42,0.41,0.41
4,0.92,0.41,0.41


## Statistiques descriptives

Une première étape consiste à comparer :
- moyennes
- distributions globales

Cela permet d’identifier rapidement des écarts majeurs.

In [5]:
reference_df.describe().T.head()

,count,mean,std,min,25%,50%,75%,max
EXT_SOURCE_1,8674.0,0.502448,0.212001,0.018668,0.333058,0.506560,0.675702,0.941621
EXT_SOURCE_2,19951.0,0.514606,0.191459,0.000010,0.395376,0.566880,0.664083,0.855000
EXT_SOURCE_3,16063.0,0.512306,0.194366,0.000527,0.372334,0.538863,0.670652,0.893976


In [6]:
current_df.describe().T.head()

,count,mean,std,min,25%,50%,75%,max
EXT_SOURCE_1,12.0,0.618333,0.244274,0.2,0.4950,0.52,0.92,0.92
EXT_SOURCE_2,12.0,0.305833,0.200066,0.1,0.1000,0.41,0.41,0.71
EXT_SOURCE_3,12.0,0.415833,0.267631,0.1,0.3325,0.41,0.41,0.91


## Détection du Data Drift avec Evidently

Nous utilisons la librairie **Evidently AI** pour :

- détecter automatiquement les dérives de distribution
- adapter les tests statistiques selon le type de variable
- produire un rapport interactif

---

### Méthodologie

Pour chaque feature :
- test statistique adapté (KS test, PSI, etc.)
- comparaison référence vs production
- classification drift / no drift

---

### Important

- Les données infinies sont remplacées par NaN
- Les colonnes non communes sont exclues
- Les données de production sont issues d’un échantillon simulé

## Génération du rapport de drift

Nous utilisons un script dédié pour générer un rapport HTML interactif.

In [9]:
from subprocess import run
from credexp.config import settings
import sys

run(
    [
        sys.executable,
        str(settings.project_root / "scripts" / "monitoring_drift.py"),
        "--limit",
        "500",
    ],
    check=True,
)

CompletedProcess(args=['g:\\Mon Drive\\OC\\Projet_6\\credexp\\.venv\\Scripts\\python.exe', 'G:\\Mon Drive\\OC\\Projet_6\\credexp\\scripts\\monitoring_drift.py', '--limit', '500'], returncode=0)

## Visualisation du rapport Evidently

Le rapport HTML contient :
- un résumé global du drift
- une analyse feature par feature
- les distributions comparées

In [ ]:
report_path = Path("reports/monitoring/evidently_drift.html")
report_path

- Ouvrir le fichier suivant dans un navigateur :

`reports/monitoring/evidently_drift.html`

## Interprétation des résultats

### Observations

- Certaines features présentent une dérive significative
- D'autres restent stables entre entraînement et production
- Le niveau de drift dépend du volume et de la nature des données

---

### Limites

- Les données de production sont simulées
- Le volume est encore faible
- Les distributions ne sont pas totalement représentatives

---

###  Risques

Le data drift peut entraîner :
- une baisse de performance du modèle
- une augmentation des erreurs métier
- des décisions incorrectes (refus/crédit)

---

###  Actions possibles

- alerter si drift dépasse un seuil
- réentraîner le modèle
- adapter les features
- monitorer en continu

## Conclusion

Ce module de monitoring permet de :

✔ détecter automatiquement la dérive des données  
✔ visualiser les écarts de distribution  
✔ anticiper une dégradation du modèle  

---

### Intégration dans l’architecture MLOps

Le système complet inclut :

- API FastAPI → prédictions temps réel
- PostgreSQL → stockage des logs
- Prometheus + Grafana → monitoring technique
- Streamlit → interface utilisateur
- Evidently → monitoring des données

---

### Perspectives

- automatiser le drift monitoring (batch régulier)
- intégrer des alertes
- coupler avec monitoring de performance modèle
- déclencher un retraining automatique

---

- Ce module constitue une brique essentielle d’un système MLOps robuste.